# ETL BBRI Market Intelligence

Notebook ini mendokumentasikan proses ETL untuk dashboard **Business Intelligence Market Intelligence & Corporate Action Support BBRI**. Notebook memakai fungsi yang sama dengan aplikasi dashboard (`etl/pipeline.py`) agar hasil validasi, transformasi, data mart, insight, dan alert tetap konsisten.

## 1. Setup Environment

Bagian ini menyiapkan path project dan mengimpor fungsi ETL yang digunakan oleh dashboard.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'etl'))

from pipeline import (
    RAW_PATH,
    read_csv,
    validate_dataframe,
    transform_dataframe,
    build_data_mart,
    summarize_metrics,
    generate_alerts,
    run_pipeline,
)

RAW_PATH

## 2. Extract

Dataset utama adalah `BBRI.csv` yang berisi data perdagangan harian saham BBRI.

In [ ]:
raw_df = read_csv(RAW_PATH)
print(f'Jumlah baris: {len(raw_df):,}')
print(f'Jumlah kolom: {len(raw_df.columns):,}')
raw_df.head()

## 3. Validate

Validasi dilakukan sebelum data masuk ke dashboard. Rule validasi meliputi kelengkapan kolom, ticker BBRI, format tanggal, tipe numerik, duplicate date, missing value kritis, `Open = 0`, volume tidak valid, outlier, dan konsistensi OHLC.

In [ ]:
validated_df, validation_report = validate_dataframe(raw_df)

validation_summary = pd.DataFrame([
    ['Status', validation_report.status],
    ['Rows', validation_report.row_count],
    ['Columns', validation_report.column_count],
    ['Min Date', validation_report.min_date],
    ['Max Date', validation_report.max_date],
    ['Duplicate Date', validation_report.duplicate_dates],
    ['Open = 0', validation_report.open_zero_count],
    ['Missing Critical', validation_report.missing_critical_count],
], columns=['Check', 'Value'])

validation_summary

In [ ]:
print('Errors:')
print(validation_report.errors if validation_report.errors else 'Tidak ada error')
print('\nWarnings:')
print(validation_report.warnings if validation_report.warnings else 'Tidak ada warning')

## 4. Transform

Transformasi menghasilkan indikator yang dibutuhkan dashboard: `Daily_Return`, `Intraday_Return`, `Net_Foreign_Flow`, `Rolling_Volatility_20D`, `Drawdown`, periode waktu, dan `Market_Status`.

In [ ]:
transformed_df = transform_dataframe(validated_df)
transformed_df[[
    'Date', 'Ticker', 'Close', 'Daily_Return', 'Net_Foreign_Flow',
    'Rolling_Volatility_20D', 'Drawdown', 'Market_Status'
]].tail(10)

## 5. Load Data Mart

Data mart sederhana dibuat agar hasil ETL siap digunakan pada tahap dashboard dan dapat dikembangkan ke DBMS/data warehouse.

In [ ]:
mart_tables = build_data_mart(transformed_df)
pd.DataFrame([
    [name, len(frame), len(frame.columns)]
    for name, frame in mart_tables.items()
], columns=['Table', 'Rows', 'Columns'])

## 6. Metrics, Insight, dan Alert

Bagian ini menghasilkan ringkasan KPI dan alert berbasis rule yang digunakan oleh dashboard actionable.

In [ ]:
metrics = summarize_metrics(transformed_df)
pd.DataFrame(metrics.items(), columns=['Metric', 'Value'])

In [ ]:
alerts = generate_alerts(transformed_df)
pd.DataFrame(alerts)

## 7. Run Full Pipeline

Cell ini menjalankan pipeline penuh seperti yang digunakan oleh script `etl/run_etl.py`. Jika `write_outputs=True`, hasilnya disimpan ke `data/processed` dan `outputs/validation`.

In [ ]:
processed_df, report, mart, pipeline_metrics, pipeline_alerts = run_pipeline(RAW_PATH, write_outputs=False)

print(f'Status validasi: {report.status}')
print(f'Rows: {report.row_count:,} | Columns: {report.column_count}')
print(f'Periode: {report.min_date} sampai {report.max_date}')
print(f'Duplicate date: {report.duplicate_dates}')
print(f'Open = 0: {report.open_zero_count}')
print(f'Data mart: {list(mart.keys())}')
print(f'Harga terakhir: {pipeline_metrics.get("latest_close")}')
print(f'Alert: {[alert["condition"] for alert in pipeline_alerts]}')